# Tech Pulse
## Does online chatter predict what a stock does next?

**Data Visualization · Final Individual Project · Summer 2026**

---

Every day millions of people argue about companies online: engineers on Hacker News, retail
traders on Reddit, journalists writing for the wire services. This project asks whether that
chatter contains information the market has not yet priced in.

Six independent sources spanning 2015 to 2026, matched to roughly two thousand publicly
traded companies and joined against daily price history.

**The short answer: yes, but modestly — and the edge has been shrinking.**

### How this notebook relates to the dashboard

Every figure here also appears in the [Streamlit dashboard](https://share.streamlit.io),
which adds sector and date filters on top. The notebook is the argument in fixed form; the
dashboard lets a reader change the inputs. Nothing is shown in one that is missing from the
other.

### Structure

Part 1 is preliminary exploration, establishing the shape and coverage of the data. Part 2
poses thirteen analytical questions, each answered with one explanatory figure.

---

## Where the data comes from

Collection notebooks live in `Python files A-D/` and `incremental_updater.ipynb`. Raw pulls
are committed to the repository so every figure below is reproducible.

| # | Source | Endpoint | Stored at |
|---|--------|----------|-----------|
| 1 | **GDELT Global Knowledge Graph** — worldwide news, indexed continuously | [`api.gdeltproject.org/api/v2/doc/doc`](https://api.gdeltproject.org/api/v2/doc/doc) · [docs](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/) | `Raw Data Pulled/gdelt_data/` |
| 2 | **Hacker News** — story titles and scores from a technical audience | [`hn.algolia.com/api/v1/search_by_date`](https://hn.algolia.com/api) | `Raw Data Pulled/hn_data/` |
| 3 | **Reddit** — r/wallstreetbets, r/stocks, r/investing, r/technology, r/SecurityAnalysis | [Arctic Shift](https://arctic-shift.photon-reddit.com/) · [source](https://github.com/ArthurHeitmann/arctic_shift) | `Raw Data Pulled/reddit_data/` |
| 4 | **Alpha Vantage + Polygon** — article sentiment pre-tagged to tickers | [AV NEWS_SENTIMENT](https://www.alphavantage.co/documentation/#news-sentiment) · [Polygon news](https://polygon.io/docs/stocks/get_v2_reference_news) | `Raw Data Pulled/stocktwits_data/` |
| 5 | **SEC EDGAR** — 8-K filings and the official ticker register | [`data.sec.gov/submissions/`](https://data.sec.gov/submissions/) · [`company_tickers.json`](https://www.sec.gov/files/company_tickers.json) | `Raw Data Pulled/edgar_data/` |
| 6 | **Daily prices** — adjusted OHLCV | [Yahoo Finance](https://finance.yahoo.com/) via [`yfinance`](https://github.com/ranaroussi/yfinance) | `stock_tracking/stocks/` |

Three details worth knowing. GDELT's `artlist` mode returns no usable tone value, so its
headlines are scored with the same sentiment model as every other text source. The
`stocktwits_data` folder name is historical — StockTwits was the original source but moved
behind Cloudflare bot protection, and the two APIs above replaced it. The SEC requires a
descriptive User-Agent on every request.

This notebook reads the **processed** outputs, not the raw pulls: quarterly signals from
`stock_tracking/sentiment_outputs/`, correlations from `correlation_outputs/`, and backtest
curves from `strategy_outputs/`. Regenerate those with notebooks A, B and C in order.

Attribute types across the merged dataset span numerical, categorical, temporal and
unstructured text.

## Setup

In [1]:
import pandas as pd
import numpy as np
import requests, io, warnings
from datetime import datetime

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)

GITHUB_REPO   = 'annhmartin/dataviz-historical-stocks-AnnetteMartin'
TRACK         = 'stock_tracking'
OUTPUT_PREFIX = f'{TRACK}/sentiment_outputs'
CORR_PREFIX   = f'{TRACK}/correlation_outputs'
STRAT_PREFIX  = f'{TRACK}/strategy_outputs'
STOCKS_PREFIX = f'{TRACK}/stocks'

print('Plotly', __import__('plotly').__version__)

Plotly 6.3.0


### Visual design system

Identical to the dashboard, defined once in `stock_tracking/utils.py` and mirrored here.

Colour carries meaning in three separate roles and the roles never share hues. **Sentiment**
uses a diverging blue-to-orange scale. **Strategies** use a categorical set where the two
passive benchmarks are muted grey so the signal-driven strategies carry the saturated colour.
**Quadrants** encode correctness by temperature — cool for correct, warm for incorrect.

Values come from the Okabe-Ito and Paul Tol qualitative schemes, both built to stay separable
under red-green colour blindness. The conventional green/red pairing is deliberately avoided:
it collapses to a single mustard tone for roughly one man in twelve, and sentiment direction
is the most important encoding in the project.

In [2]:
# Sentiment, diverging
POS, NEG, NEU = '#0072B2', '#D55E00', '#AEB4BF'
# Highlights and context
ACCENT, HIGHLIGHT, GOLD = '#009E73', '#CC79A7', '#E69F00'
CONTEXT, INK, MUTED, GRIDLINE = '#BFC5CF', '#22262E', '#6B7280', '#EDEFF3'
PRICE, CANVAS = '#4A4E57', '#FFFFFF'
# Quadrants: cool = correct, warm = incorrect
QUAD_TP, QUAD_TN, QUAD_FP, QUAD_FN = '#0072B2', '#009E73', '#D55E00', '#CC79A7'
# Nine sources, nine distinct hues
SOURCE_COLORS = {
    'gdelt':'#0072B2', 'hn':'#E69F00', 'stocktwits':'#009E73', 'edgar_8k':'#332288',
    'reddit_wallstreetbets':'#D55E00', 'reddit_stocks':'#88CCEE',
    'reddit_investing':'#999933', 'reddit_technology':'#AA4499',
    'reddit_SecurityAnalysis':'#661100'}

SENTIMENT_THRESHOLD = 0.05

SECTOR_MAP = {
    'AI Accelerators'      : ['NVDA','AMD'],
    'Semiconductor Supply' : ['TSM','INTC','QCOM'],
    'Big Tech'             : ['GOOGL','MSFT','AAPL','META'],
    'Cloud / SaaS'         : ['AMZN','SNOW','DDOG','CRM','NOW','MDB'],
    'Cybersecurity'        : ['CRWD','PANW','OKTA'],
    'Enterprise AI'        : ['PLTR'],
    'Macro Risk'           : ['COIN','TSLA'],
    'Portfolio'            : ['INCY','KGC','NVO','PM','WPM'],
    'Consumer Tech'        : ['NFLX','SPOT','PINS'],
    'Enterprise Fintech'   : ['PYPL'],
}
TICKER_TO_SECTOR = {t: s for s, ts in SECTOR_MAP.items() for t in ts}
ALL_TICKERS = sorted(TICKER_TO_SECTOR)

pio.templates['techpulse'] = go.layout.Template(layout=go.Layout(
    font=dict(family='Helvetica Neue, Helvetica, Arial, sans-serif', size=14, color=INK),
    title=dict(font=dict(size=20, color=INK), x=0.01, xanchor='left', y=0.95, yanchor='top'),
    paper_bgcolor=CANVAS, plot_bgcolor=CANVAS,
    xaxis=dict(showgrid=False, zeroline=False, showline=True, linecolor=GRIDLINE,
               ticks='outside', tickcolor=GRIDLINE, ticklen=5,
               tickfont=dict(color=MUTED, size=12),
               title=dict(font=dict(size=13, color=MUTED))),
    yaxis=dict(showgrid=True, gridcolor=GRIDLINE, zeroline=False, showline=False,
               tickfont=dict(color=MUTED, size=12),
               title=dict(font=dict(size=13, color=MUTED))),
    legend=dict(bgcolor='rgba(0,0,0,0)', borderwidth=0, font=dict(size=12, color=INK)),
    margin=dict(l=70, r=40, t=110, b=70),
    hoverlabel=dict(bgcolor='white', font_size=13, bordercolor=GRIDLINE),
    colorway=[POS, NEG, ACCENT, HIGHLIGHT, GOLD, MUTED]))
pio.templates.default = 'techpulse'

def titled(fig, takeaway, subtitle=None, height=520):
    """Title states the finding; subtitle carries the mechanics."""
    text = f'<b>{takeaway}</b>'
    if subtitle:
        text += f"<br><span style='font-size:13px;color:{MUTED}'>{subtitle}</span>"
    fig.update_layout(title=dict(text=text), height=height)
    return fig

def note(fig, text, x, y, ax=0, ay=-40, color=None):
    fig.add_annotation(x=x, y=y, text=text, showarrow=True, arrowhead=0,
                       arrowcolor=color or MUTED, arrowwidth=1.2, ax=ax, ay=ay,
                       font=dict(size=12, color=color or INK), align='left',
                       bgcolor='rgba(255,255,255,0.85)', borderpad=4)
    return fig

print('Design system loaded')

Design system loaded


## Loading the data

In [3]:
def load_csv(path, token=None):
    safe = path.replace(' ', '%20')
    url  = f'https://raw.githubusercontent.com/{GITHUB_REPO}/main/{safe}'
    hdrs = {'Authorization': f'Bearer {token}'} if token else {}
    r = requests.get(url, headers=hdrs, timeout=90)
    if r.status_code != 200 or not r.text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(r.text), low_memory=False)

TOKEN = None   # set if the repository is private

print('Loading sentiment signals ...')
frames = []
for year in range(2015, datetime.now().year + 1):
    for q in (1, 2, 3, 4):
        df = load_csv(f'{OUTPUT_PREFIX}/daily_signals_{year}_Q{q}.csv', TOKEN)
        if not df.empty:
            keep = [c for c in ['ticker','date','norm_sentiment','adaptive_sentiment',
                                'story_count','sources_active'] if c in df.columns]
            frames.append(df[keep])
signals = pd.concat(frames, ignore_index=True)
signals['date'] = pd.to_datetime(signals['date'])
SIG = 'adaptive_sentiment' if 'adaptive_sentiment' in signals.columns else 'norm_sentiment'
print(f'  {len(signals):,} ticker-days across {signals.ticker.nunique():,} tickers')

print('Loading prices ...')
pframes = []
for t in ALL_TICKERS:
    p = load_csv(f'{STOCKS_PREFIX}/prices_{t}.csv', TOKEN)
    if p.empty or 'Close' not in p.columns:
        continue
    p = p[['Date','Close']].copy()
    p['Date']   = pd.to_datetime(p['Date'])
    p['ticker'] = t
    p = p.sort_values('Date')
    p['ret_1d'] = p['Close'].pct_change(fill_method=None)
    p['pct_7d'] = p['Close'].pct_change(7, fill_method=None) * 100
    pframes.append(p)
prices = pd.concat(pframes, ignore_index=True)
print(f'  {len(prices):,} price rows across {prices.ticker.nunique()} tickers')

source_attr = load_csv(f'{OUTPUT_PREFIX}/source_attribution.csv', TOKEN)
best_corr   = load_csv(f'{CORR_PREFIX}/best_per_ticker.csv', TOKEN)
key_moves   = load_csv(f'{CORR_PREFIX}/key_moves.csv', TOKEN)
equity      = load_csv(f'{STRAT_PREFIX}/equity_curves.csv', TOKEN)
trades      = load_csv(f'{STRAT_PREFIX}/trade_log.csv', TOKEN)
print(f'  source_attr {len(source_attr):,} | best_corr {len(best_corr):,} | '
      f'key_moves {len(key_moves):,} | equity {len(equity):,} | trades {len(trades):,}')

Loading sentiment signals ...
  8,233,536 ticker-days across 1,952 tickers
Loading prices ...
  94,412 price rows across 27 tickers
  source_attr 5,934 | best_corr 1,506 | key_moves 23,309 | equity 3,497 | trades 218


## Shared helper

Every question below judges a sentiment reading the same way: take the days when sentiment
took a clear stance, look at what price did over the following few trading days, and record
whether the direction matched. Defining that once keeps the questions comparable.

In [4]:
HOLD = 5   # trading days over which a signal is judged

def evaluate_signals(tickers=None, hold=HOLD, min_strength=SENTIMENT_THRESHOLD):
    """
    Join sentiment readings to the forward return that followed them.
    Returns one row per signal with its direction, the return, and whether
    the call was correct.
    """
    tickers = tickers or ALL_TICKERS
    out = []
    for ticker in tickers:
        p = prices[prices.ticker == ticker]
        if p.empty:
            continue
        p = p.sort_values('Date')
        closes, pdates = p['Close'].values, p['Date'].values
        s = signals[(signals.ticker == ticker) & signals[SIG].notna()]
        s = s[s[SIG].abs() >= min_strength].sort_values('date')
        if s.empty:
            continue
        entry = np.searchsorted(pdates, s['date'].values, side='right')
        ex    = entry + hold - 1
        ok    = (ex < len(closes)) & (entry < len(closes))
        if ok.sum() == 0:
            continue
        out.append(pd.DataFrame({
            'ticker' : ticker,
            'sector' : TICKER_TO_SECTOR.get(ticker),
            'date'   : s['date'].values[ok],
            'sent'   : s[SIG].values[ok],
            'stories': s['story_count'].values[ok] if 'story_count' in s.columns else 1,
            'ret'    : (closes[ex[ok]] - closes[entry[ok]]) / closes[entry[ok]] * 100,
        }))
    ev = pd.concat(out, ignore_index=True)
    ev['correct'] = ((ev.sent > 0) & (ev.ret > 0)) | ((ev.sent < 0) & (ev.ret < 0))
    ev['dir']     = np.where(ev.sent > 0, 'Positive', 'Negative')
    ev['year']    = pd.to_datetime(ev.date).dt.year
    return ev

ev_all = evaluate_signals()
print(f'{len(ev_all):,} signals evaluated over a {HOLD}-day horizon')
print(f'Overall directional accuracy: {ev_all.correct.mean():.1%}')

39,167 signals evaluated over a 5-day horizon
Overall directional accuracy: 51.9%


---

## Part 1 — Preliminary exploration

Descriptive by design. These establish the shape of the data and do not count toward the
analytical questions.

In [5]:
print('SHAPE')
print(f'  Signal rows         {len(signals):,}')
print(f'  Tickers with signal {signals.ticker.nunique():,}')
print(f'  Date range          {signals.date.min():%Y-%m-%d} to {signals.date.max():%Y-%m-%d}')
print(f'  Days covered        {(signals.date.max() - signals.date.min()).days:,}')

print('\nSENTIMENT DISTRIBUTION')
print(signals[SIG].describe().round(4).to_string())

print('\nMISSINGNESS (%)')
miss = (signals.isna().sum() / len(signals) * 100).round(2)
print(miss[miss > 0].to_string() if (miss > 0).any() else '  none')

print('\nCOVERAGE BY SECTOR')
cov = (signals[signals.ticker.isin(ALL_TICKERS)]
       .assign(sector=lambda d: d.ticker.map(TICKER_TO_SECTOR))
       .groupby('sector')
       .agg(tickers=('ticker','nunique'),
            signal_days=(SIG, lambda x: int(x.notna().sum())),
            stories=('story_count','sum'))
       .sort_values('stories', ascending=False))
print(cov.to_string())

SHAPE
  Signal rows         8,233,536
  Tickers with signal 1,952
  Date range          2015-01-01 to 2026-07-19
  Days covered        4,217

SENTIMENT DISTRIBUTION
count    866021.0000
mean          0.1358
std           0.3082
min          -0.5866
25%           0.0000
50%           0.0000
75%           0.2493
max           1.4742

MISSINGNESS (%)
norm_sentiment        98.38
adaptive_sentiment    89.48

COVERAGE BY SECTOR
                      tickers  signal_days  stories
sector                                             
Big Tech                    4        16381   113401
Macro Risk                  2         8198    56860
Cloud / SaaS                6        17305    41724
AI Accelerators             2         8127    40715
Cybersecurity               3         9580    31687
Semiconductor Supply        3        10215    30446
Consumer Tech               2         7914    18447
Enterprise Fintech          1         3970    15239
Enterprise AI               1         3541    12375
Po

In [6]:
# Coverage is heavily skewed, which shapes how every later result must be read
per_ticker = signals.groupby('ticker')['story_count'].sum()
per_ticker = per_ticker[per_ticker > 0]

fig = go.Figure(go.Histogram(
    x=np.log10(per_ticker), nbinsx=50, marker_color=CONTEXT,
    hovertemplate='10^%{x:.1f} stories<br>%{y} tickers<extra></extra>'))
med = float(per_ticker.median())
fig.add_vline(x=np.log10(med), line_color=NEG, line_width=2,
              annotation_text=f'median {med:,.0f}', annotation_position='top right',
              annotation_font=dict(color=NEG, size=12))
fig.update_xaxes(title='Total stories per ticker (log₁₀ scale)')
fig.update_yaxes(title='Number of tickers')
titled(fig, 'Coverage is extremely uneven across the ticker universe',
       'A handful of companies attract most of the discussion; the long tail is thin and '
       'produces noisier signals', height=440)
fig.show()

---

# Part 2 — Analytical questions

Thirteen questions, each answered with one explanatory figure. Every figure below also
appears in the dashboard, where sector and date filters make it interactive.

---

## Q1 · Is positive or negative sentiment the more reliable signal?

*Dashboard: Key Findings, figure 1*

Financial commentary skews optimistic, so enthusiasm is cheap while criticism is
comparatively rare. If negativity costs more to express it may carry more information when
it appears. Splitting hit rate by direction tests that.

In [7]:
by_dir = ev_all.groupby('dir')['correct'].agg(['mean','size'])
pos_rate = float(by_dir.loc['Positive','mean']); pos_n = int(by_dir.loc['Positive','size'])
neg_rate = float(by_dir.loc['Negative','mean']); neg_n = int(by_dir.loc['Negative','size'])
print(by_dir.round(4).to_string())

fig = go.Figure()
for label, rate, n, colour in [('Positive buzz', pos_rate, pos_n, POS),
                               ('Negative buzz', neg_rate, neg_n, NEG)]:
    fig.add_trace(go.Bar(x=[label], y=[rate], marker_color=colour, marker_line_width=0,
                         width=0.45, text=[f'{rate*100:.1f}%'], textposition='outside',
                         textfont=dict(size=17, color=colour), customdata=[n],
                         hovertemplate='<b>%{x}</b><br>correct %{y:.1%}'
                                       '<br>%{customdata:,} signals<extra></extra>'))
fig.add_hline(y=0.5, line_color=MUTED, line_width=2, line_dash='dot')
fig.add_annotation(x=1.45, y=0.5, text='coin flip', showarrow=False, xanchor='right',
                   yshift=13, font=dict(size=13, color=MUTED))
fig.update_yaxes(title='Share of signals that called direction correctly', tickformat='.0%',
                 range=[min(0.35, neg_rate-0.08), max(0.62, pos_rate+0.08)])
fig.update_layout(showlegend=False)
titled(fig, f'Does positive buzz beat chance? Yes, by {(pos_rate-0.5)*100:.1f} points',
       f'Judged over the following {HOLD} trading days. Negative signals are shown on a much '
       f'smaller sample ({neg_n:,} against {pos_n:,})', height=440)
fig.show()

            mean   size
dir                    
Negative  0.4567   9070
Positive  0.5379  30097


**Finding.** Positive sentiment clears chance by a small but consistent margin across
roughly thirty thousand signals. The margin is modest by necessity — an obvious edge would
already have been arbitraged away.

Negative signals came in *below* chance, which would imply prices rose after negative
chatter. That contradicts a well-established literature on negative news and rests on a
sample around a third the size, so it is treated as a limitation of the measurement rather
than a tradeable finding. Three explanations are more plausible than a genuine reversal: the
smaller sample makes the estimate less stable; VADER is a general-purpose model that misreads
financial phrasing and has no way of knowing what "missed estimates" implies; and scoring
headlines rather than full articles strips the context separating real bad news from routine
commentary. The strategies later on lean on positive signals accordingly.

---

## Q2 · Has the edge faded over time?

*Dashboard: Key Findings, figure 2*

Between 2015 and 2026 commission-free trading arrived, retail participation surged, and
quantitative funds began mining social data directly. If sentiment carried a real edge,
competition should have eroded it.

In [8]:
by_year = (ev_all.groupby('year')['correct'].agg(['mean','size'])
           .reset_index().rename(columns={'mean':'hit','size':'n'}))
by_year = by_year[by_year.n >= 100]
early = float(by_year[by_year.year < 2021].hit.mean())
late  = float(by_year[by_year.year >= 2021].hit.mean())
print(by_year.round(4).to_string(index=False))
print(f'\npre-2021 {early:.1%} | 2021 onward {late:.1%} | change {(late-early)*100:+.1f} pts')

colours = [CONTEXT if y < 2021 else POS for y in by_year.year]
fig = go.Figure(go.Bar(x=by_year.year, y=by_year.hit, marker_color=colours,
                       marker_line_width=0, customdata=by_year.n,
                       hovertemplate='%{x}<br>accuracy %{y:.1%}'
                                     '<br>%{customdata:,} signals<extra></extra>'))
fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')
fig.add_hline(y=early, line_color=CONTEXT, line_width=2, line_dash='dash')
fig.add_hline(y=late,  line_color=POS,     line_width=2, line_dash='dash')
fig.add_annotation(x=by_year.year.min(), y=early, text=f'pre-2021 average {early:.1%}',
                   showarrow=False, xanchor='left', yshift=13, font=dict(size=12, color=MUTED))
fig.add_annotation(x=by_year.year.max(), y=late, text=f'2021 onward {late:.1%} ',
                   showarrow=False, xanchor='right', yshift=-15, font=dict(size=12, color=POS))
fig.update_yaxes(title='Directional accuracy', tickformat='.0%')
fig.update_xaxes(title='', dtick=1)
fig.update_layout(showlegend=False)
titled(fig, f'Has the edge faded? Accuracy fell {abs(late-early)*100:.1f} points after 2021',
       'Annual hit rate. Years with fewer than 100 signals are excluded', height=450)
fig.show()

 year    hit    n
 2015 0.4561 3328
 2016 0.5240 3420
 2017 0.5506 3182
 2018 0.5187 3451
 2019 0.5545 3358
 2020 0.5324 3210
 2021 0.5208 3335
 2022 0.4597 3285
 2023 0.5321 3920
 2024 0.5331 3892
 2025 0.5366 3388
 2026 0.4871 1398

pre-2021 52.3% | 2021 onward 51.2% | change -1.1 pts


**Finding.** The signal is not stationary. Accuracy was measurably higher before 2021 than
after, which is consistent with an edge being competed away as more participants traded on
the same public signals. Any backtest quoting a single average across eleven years conceals
this, which is why Q10 shows where the strategy gained and lost rather than only a final
figure.

---

## Q3 · Would trading these signals have beaten the market?

*Dashboard: Key Findings figure 3, and the Strategies page*

Everything above measures whether the signal contains information. This asks whether that
information survives contact with execution — holding periods, transaction costs, and
committing capital before knowing the outcome. The lower panel is the honest part: a single
final number hides how the return was earned.

In [9]:
eq = equity.copy()
eq['date'] = pd.to_datetime(eq['date'])
curves = eq.set_index('date').sort_index().ffill()

SERIES = [('S&P_500_SPY',     'S&P 500 (SPY)',   CONTEXT,   'dash',  2.0),
          ('Buy_&_Hold',      'Buy & Hold',      MUTED,     'solid', 2.0),
          ('Sector_Rotation', 'Sector Rotation', ACCENT,    'solid', 3.2),
          ('Position_Trader', 'Position Trader', HIGHLIGHT, 'solid', 2.2)]
SERIES = [s for s in SERIES if s[0] in curves.columns]
finals = {l: float(curves[cc].dropna().iloc[-1]) for cc, l, *_ in SERIES}
for k, v in sorted(finals.items(), key=lambda x: -x[1]):
    print(f'  {k:<20} ${v:>12,.0f}   {(v/10000-1)*100:>8.1f}%')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.10,
                    row_heights=[0.68, 0.32],
                    subplot_titles=('', 'Rolling 12-month margin over the S&P 500'))
for cc, label, colour, dash, width in SERIES:
    s = curves[cc].dropna()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines', name=label,
                             line=dict(color=colour, width=width, dash=dash),
                             hovertemplate=f'<b>{label}</b><br>%{{x|%b %Y}}'
                                           f'<br>$%{{y:,.0f}}<extra></extra>'), row=1, col=1)

behind = np.nan
if {'Sector_Rotation','S&P_500_SPY'} <= set(curves.columns):
    r12 = curves.pct_change(252)
    margin = (r12['Sector_Rotation'] - r12['S&P_500_SPY']).dropna()
    behind = float((margin < 0).mean())
    fig.add_trace(go.Scatter(x=margin.index, y=margin.values, mode='lines',
                             line=dict(color=ACCENT, width=2),
                             name='Rotation minus S&P 500',
                             hovertemplate='%{x|%b %Y}<br>%{y:+.1%}<extra></extra>'),
                  row=2, col=1)
    fig.add_trace(go.Scatter(x=margin.index,
                             y=np.where(margin.values < 0, margin.values, 0),
                             fill='tozeroy', mode='none', fillcolor='rgba(213,94,0,0.28)',
                             name='Trailing the index', hoverinfo='skip'), row=2, col=1)
    fig.add_hline(y=0, line_color=MUTED, line_width=1.5, row=2, col=1)

fig.update_yaxes(title='Portfolio value', tickprefix='$', row=1, col=1)
fig.update_yaxes(title='Margin', tickformat='+.0%', row=2, col=1)
fig.update_layout(hovermode='x unified', margin=dict(t=150),
                  legend=dict(orientation='h', y=1.03, x=0, yanchor='bottom',
                              itemwidth=40, itemsizing='constant'))
best = max(finals, key=finals.get)
spy  = finals.get('S&P 500 (SPY)')
sub  = f'From $10,000 in 2015, including transaction costs'
if spy: sub += f'. Best strategy reached {finals[best]/spy:.1f}× the index'
titled(fig, f'Which strategy came out ahead? {best}', sub, height=720)
fig.show()
if behind == behind:
    print(f'\nSector Rotation trailed the index in {behind:.1%} of rolling 12-month windows.')

  Sector Rotation      $     141,694     1316.9%
  Position Trader      $      85,350      753.5%
  Buy & Hold           $      72,005      620.1%
  S&P 500 (SPY)        $      43,868      338.7%



Sector Rotation trailed the index in 43.7% of rolling 12-month windows.


**Finding.** Both sentiment-driven strategies finish ahead of the passive benchmarks after
costs. The lower panel is the necessary caveat: outperformance arrives in bursts separated by
long stretches of trailing the index. An investor would have had to hold through extended
underperformance to collect the final result, which is a materially different experience from
what the headline number implies.

---

## Q4 · Which strategy leads in any given month?

*Dashboard: Strategies page, second figure*

The equity curves show who won overall. This shows whether leadership was stable or rotated —
and marks the months when every strategy lost money, which no single curve reveals.

In [10]:
monthly = curves[[s[0] for s in SERIES]].resample('ME').last().dropna(how='all')
rets    = monthly.pct_change()
winners = monthly.idxmax(axis=1)
label_for = {cc: l for cc, l, *_ in SERIES}
color_for = {cc: col for cc, _, col, _, _ in SERIES}

kinds = []
for i, (month, win) in enumerate(winners.items()):
    if i > 0 and month in rets.index and rets.loc[month].notna().any() \
       and (rets.loc[month].dropna() < 0).all():
        kinds.append('all lost')
    else:
        kinds.append(label_for.get(win, win))

fig = go.Figure()
for cc, label, colour, _d, _w in SERIES:
    mask = [k == label for k in kinds]
    if not any(mask):
        continue
    fig.add_trace(go.Bar(x=winners.index[mask], y=[1]*sum(mask), name=label,
                         marker_color=colour, marker_line_width=0,
                         hovertemplate='%{x|%b %Y}<br>'+label+'<extra></extra>'))
mask_none = [k == 'all lost' for k in kinds]
if any(mask_none):
    fig.add_trace(go.Bar(x=winners.index[mask_none], y=[1]*sum(mask_none),
                         name='Every strategy lost', marker_color=CONTEXT,
                         marker_line_width=0,
                         hovertemplate='%{x|%b %Y}<br>every strategy lost<extra></extra>'))

fig.update_yaxes(showticklabels=False, showgrid=False, title='')
fig.update_layout(barmode='stack', bargap=0.05, margin=dict(b=120),
                  legend=dict(orientation='h', yanchor='top', y=-0.35, x=0))
lost_share = sum(mask_none) / len(kinds)
titled(fig, 'Which strategy was ahead in any given month?',
       f'Grey marks months where every strategy lost money — {lost_share:.0%} of the period',
       height=330)
fig.show()

**Finding.** Leadership rotates rather than settling. The grey months matter as much as the
coloured ones: they are periods when no approach protected capital, which a cumulative curve
smooths out of view entirely.

---

## Q5 · What would a specific investment have returned?

*Dashboard: What If page, first figure*

The same backtest rescaled to a chosen starting amount. In the dashboard the reader sets the
figure; here it is fixed at $10,000 to match the benchmark comparison above.

In [11]:
AMOUNT = 10_000
fig = go.Figure()
scaled_finals = {}
for cc, label, colour, dash, width in SERIES:
    s = curves[cc].dropna()
    scaled = s * (AMOUNT / float(s.iloc[0]))
    scaled_finals[label] = float(scaled.iloc[-1])
    fig.add_trace(go.Scatter(x=scaled.index, y=scaled.values, mode='lines', name=label,
                             line=dict(color=colour, width=width, dash=dash),
                             hovertemplate=f'<b>{label}</b><br>%{{x|%b %Y}}'
                                           f'<br>$%{{y:,.0f}}<extra></extra>'))
    fig.add_annotation(x=scaled.index[-1], y=float(scaled.iloc[-1]),
                       text=f'  ${float(scaled.iloc[-1]):,.0f}', showarrow=False,
                       xanchor='left', font=dict(size=12, color=colour))
fig.add_hline(y=AMOUNT, line_color=MUTED, line_width=1.5, line_dash='dot')
fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                         line=dict(color=MUTED, width=2, dash='dot'),
                         name=f'Starting amount (${AMOUNT:,.0f})'))
fig.update_yaxes(title='Portfolio value', tickprefix='$')
fig.update_layout(hovermode='x unified', margin=dict(r=150, t=140),
                  legend=dict(orientation='h', y=1.03, x=0, yanchor='bottom',
                              itemwidth=40, itemsizing='constant'))
bw = max(scaled_finals, key=scaled_finals.get)
titled(fig, f'What would ${AMOUNT:,.0f} have become? ${scaled_finals[bw]:,.0f} in {bw}',
       'Every curve rescaled to the same starting amount, including transaction costs',
       height=560)
fig.show()
for k, v in sorted(scaled_finals.items(), key=lambda x: -x[1]):
    print(f'  {k:<20} ${v:>12,.0f}   {(v/AMOUNT-1)*100:>+8.1f}%')

  Sector Rotation      $     141,694    +1316.9%
  Position Trader      $      85,350     +753.5%
  Buy & Hold           $      72,200     +622.0%
  S&P 500 (SPY)        $      43,891     +338.9%


**Finding.** Framing the same result in dollars rather than percentages makes the spread
between approaches concrete, and makes clear that the gap between the best strategy and the
index compounds substantially over eleven years.

---

## Q6 · How many individual trades actually made money?

*Dashboard: What If page, second figure*

An equity curve can rise on a handful of outsized winners while most trades lose. This shows
the distribution behind the curve.

In [12]:
t = trades.copy()
t['entry_date'] = pd.to_datetime(t['entry_date'])
wins, losses = t[t.return_pct > 0], t[t.return_pct <= 0]

fig = go.Figure()
for d, colour, name in [(wins, POS, 'Winning trade'), (losses, NEG, 'Losing trade')]:
    if d.empty:
        continue
    fig.add_trace(go.Bar(x=d.entry_date, y=d.return_pct, name=name,
                         marker_color=colour, marker_line_width=0,
                         customdata=d.ticker if 'ticker' in d.columns else None,
                         hovertemplate='%{customdata}<br>%{x|%d %b %Y}'
                                       '<br>%{y:+.1f}%<extra></extra>'))
fig.add_hline(y=0, line_color=MUTED, line_width=1.5)
fig.update_yaxes(title='Trade return (%)')
fig.update_layout(bargap=0.3,
                  legend=dict(orientation='h', y=1.03, x=0, yanchor='bottom'))
wr = len(wins) / len(t) * 100
titled(fig, f'How many trades actually made money? {len(wins)} of {len(t)}, a {wr:.0f}% win rate',
       f'Average return {t.return_pct.mean():+.2f}% per trade · '
       f'best {t.return_pct.max():+.1f}% · worst {t.return_pct.min():+.1f}%', height=450)
fig.show()

**Finding.** The win rate sits close to the coin-flip line while the strategy still finishes
ahead, which tells us the return comes from asymmetry in *size* rather than frequency —
winners run further than losers fall. That is a fragile property: it depends on a small number
of large gains, so the strategy is more exposed to a single bad outcome than the aggregate
curve suggests.

---

## Q7 · When sentiment fires for one company, how often is it right?

*Dashboard: Signal Quality page*

The aggregate hit rate hides the structure of the errors. Splitting every signal into four
quadrants separates the two ways of being right from the two ways of being wrong — which
matters, because a signal that is reliable on the upside but useless on the downside needs to
be traded differently from one that works both ways.

In [13]:
QUAD_TICKER = 'NVDA'      # change to any ticker in ALL_TICKERS

d = ev_all[ev_all.ticker == QUAD_TICKER].copy()
d['quad'] = np.where((d.sent > 0) & (d.ret > 0),  'True positive',
            np.where((d.sent > 0) & (d.ret <= 0), 'False positive',
            np.where((d.sent < 0) & (d.ret < 0),  'True negative', 'False negative')))
QC = {'True positive': QUAD_TP, 'True negative': QUAD_TN,
      'False positive': QUAD_FP, 'False negative': QUAD_FN}

counts = d.quad.value_counts()
tp, fp = int(counts.get('True positive',0)),  int(counts.get('False positive',0))
tn, fn = int(counts.get('True negative',0)),  int(counts.get('False negative',0))
total  = len(d); acc = (tp + tn) / total
print(f'{QUAD_TICKER}: accuracy {acc:.1%} on {total:,} signals')
print(f'  TP {tp} | FP {fp} | TN {tn} | FN {fn}')

fig = go.Figure()
for q in ['True positive','True negative','False positive','False negative']:
    sub = d[d.quad == q]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub.sent, y=sub.ret, mode='markers', name=q,
        marker=dict(color=QC[q], size=np.clip(sub.stories, 1, 40)*0.7 + 7,
                    line=dict(color='white', width=1), opacity=0.75),
        customdata=np.stack([sub.stories, pd.to_datetime(sub.date).dt.strftime('%d %b %Y')],
                            axis=-1),
        hovertemplate='<b>'+q+'</b><br>%{customdata[1]}<br>sentiment %{x:.3f}'
                      '<br>return %{y:+.1f}%<br>%{customdata[0]} stories<extra></extra>'))

xmax = float(d.sent.abs().max())*1.12; ymax = float(d.ret.abs().max())*1.12
fig.add_hline(y=0, line_color=MUTED, line_width=1.5)
fig.add_vline(x=0, line_color=MUTED, line_width=1.5)
z  = np.polyfit(d.sent, d.ret, 1); xs = np.linspace(-xmax, xmax, 60)
fig.add_trace(go.Scatter(x=xs, y=np.poly1d(z)(xs), mode='lines',
                         line=dict(color=GOLD, width=2.5), name='Overall trend',
                         hoverinfo='skip'))
for txt, xa, ya, col in [(f'{tp} ({tp/total*100:.0f}%)',  xmax*.62,  ymax*.88, QUAD_TP),
                         (f'{fn} ({fn/total*100:.0f}%)', -xmax*.62,  ymax*.88, QUAD_FN),
                         (f'{fp} ({fp/total*100:.0f}%)',  xmax*.62, -ymax*.88, QUAD_FP),
                         (f'{tn} ({tn/total*100:.0f}%)', -xmax*.62, -ymax*.88, QUAD_TN)]:
    fig.add_annotation(x=xa, y=ya, text=f'<b>{txt}</b>', showarrow=False,
                       font=dict(size=15, color=col))
fig.update_xaxes(title='Negative buzz  ←     |     →  Positive buzz', range=[-xmax, xmax])
fig.update_yaxes(title=f'Price change {HOLD} days later (%)', range=[-ymax, ymax])
fig.update_layout(legend=dict(orientation='h', y=1.03, x=0, yanchor='bottom'))
verdict = 'beats' if acc > 0.5 else 'trails'
titled(fig, f'Does {QUAD_TICKER} sentiment beat a coin flip? It {verdict} it, at {acc*100:.1f}%',
       f'Cool colours mark correct calls, warm incorrect. Dot size is stories that day. '
       f'n = {total:,}', height=640)
fig.show()

NVDA: accuracy 54.5% on 2,359 signals
  TP 1172 | FP 872 | TN 113 | FN 202


**Finding.** The four quadrants are not evenly populated. Reading them together shows whether
accuracy comes from catching rises, avoiding falls, or both — and the trend line gives the
overall relationship in a single stroke. The dashboard lets any ticker and holding period be
substituted here.

---

## Q8 · Which companies have the most predictive chatter?

*Dashboard: Correlation page, first figure*

Ranking tickers by how strongly sentiment leads their price identifies where the signal is
worth acting on. A negative bar is not a weak result but an inverted one, and is flagged as
such.

In [14]:
MIN_N = 200
q = best_corr.copy()
if 'n' in q.columns:
    q = q[q.n >= MIN_N]
q = q.dropna(subset=['corr'])
plot_df = q[q.ticker.isin(ALL_TICKERS)].sort_values('corr')
if plot_df.empty:
    plot_df = (pd.concat([q.nsmallest(15,'corr'), q.nlargest(15,'corr')])
               .drop_duplicates('ticker').sort_values('corr'))

plot_df = plot_df.assign(
    label=plot_df.ticker + '  ' + plot_df.signal + ' T+' + plot_df.horizon.astype(str))
strongest = plot_df['corr'].abs().idxmax()

fig = go.Figure()
for positive, colour, name in [
        (True,  POS, 'Follows the buzz — positive sentiment led to gains'),
        (False, NEG, 'Contrarian — positive sentiment led to losses')]:
    sub = plot_df[(plot_df['corr'] > 0) == positive]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(y=sub.label, x=sub['corr'], orientation='h', name=name,
                         marker_color=colour, marker_line_width=0,
                         customdata=sub.n if 'n' in sub.columns else None,
                         hovertemplate='<b>%{y}</b><br>correlation %{x:+.4f}'
                                       + ('<br>%{customdata:,} observations'
                                          if 'n' in sub.columns else '')
                                       + '<extra></extra>'))
fig.add_vline(x=0, line_color=MUTED, line_width=1.5)
top = plot_df.loc[strongest]
fig.add_annotation(x=float(top['corr']), y=top.label, text='  strongest  ', showarrow=False,
                   xanchor='left' if top['corr'] > 0 else 'right',
                   font=dict(size=12, color=POS if top['corr'] > 0 else NEG))
fig.update_xaxes(title='Correlation between sentiment and forward return', tickformat='+.3f')
fig.update_yaxes(title='')
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=-0.22, x=0),
                  margin=dict(b=130))
titled(fig, f"Which company has the most predictive chatter? {top.ticker}",
       f'Only tickers with at least {MIN_N} observations are shown',
       height=max(430, len(plot_df)*26 + 230))
fig.show()
print(plot_df[['ticker','signal','horizon','corr'] +
              (['n'] if 'n' in plot_df.columns else [])].to_string(index=False))

ticker             signal  horizon    corr    n
   WPM adaptive_sentiment       10 -0.1767  209
  OKTA           roll_21d       21 -0.1309 2873
   NVO            roll_5d        2 -0.1280  585
  PLTR           roll_21d        5 -0.1244 3676
   MDB            roll_5d        2 -0.1026 2552
  CRWD           roll_21d       10 -0.0962 3678
  INCY      sent_momentum        1 -0.0914  751
    PM            roll_7d       21 -0.0785  536
 GOOGL      sent_momentum        7 -0.0665 1875
  PYPL            roll_3d       10 -0.0664 1937
  INTC      sent_momentum       10 -0.0655 2913
   CRM      sent_momentum        3 -0.0595 1549
   AMD      sent_momentum        3 -0.0564 2530
  AMZN      sent_momentum        2 -0.0555 3277
  NFLX           roll_21d        2 -0.0467 4184
  AAPL           roll_21d       21 -0.0324 4175
  TSLA           roll_21d       10  0.0324 4179
  META     norm_sentiment        5  0.0444 2546
  MSFT            roll_5d        3  0.0481 4168
  COIN            roll_7d        1  0.05

**Finding.** Predictive strength varies widely between companies even within the same sector,
and the correlations are small in absolute terms throughout — consistent with a real but
weak effect rather than a strong one. The requirement of at least 200 observations matters
here: without it, thinly covered tickers produce spuriously perfect correlations on a dozen
data points.

---

## Q9 · Has sentiment toward a company shifted over time?

*Dashboard: Correlation page, second figure*

Aggregate sentiment can look stable while the underlying mood swings. A rolling measure of
how much of the conversation is positive shows whether a company's reputation among retail
and technical audiences has drifted.

In [15]:
TS_TICKER = 'NVDA'

s = signals[(signals.ticker == TS_TICKER) & signals[SIG].notna()].sort_values('date').copy()

def positive_share(x):
    active = x[np.abs(x) > SENTIMENT_THRESHOLD]
    return 0.5 if len(active) == 0 else float((active > SENTIMENT_THRESHOLD).sum())/len(active)

s['roll'] = s[SIG].rolling(90, min_periods=20).apply(positive_share, raw=True)
d = s.dropna(subset=['roll'])

fig = go.Figure()
fig.add_trace(go.Scatter(x=d.date, y=d.roll, mode='lines',
                         line=dict(color=POS, width=2.5), name='90-day positive share',
                         hovertemplate='%{x|%b %Y}<br>%{y:.1%} positive<extra></extra>'))
fig.add_trace(go.Scatter(x=d.date, y=np.where(d.roll < 0.5, d.roll, 0.5),
                         fill='tonexty', mode='none', fillcolor='rgba(213,94,0,0.18)',
                         showlegend=False, hoverinfo='skip'))
fig.add_hline(y=0.5, line_color=MUTED, line_width=1.5, line_dash='dot')
fig.add_annotation(x=d.date.max(), y=0.5, text='even split ', showarrow=False,
                   xanchor='right', yshift=11, font=dict(size=12, color=MUTED))
fig.update_yaxes(title='Share of active signals that were positive',
                 tickformat='.0%', range=[0,1])
recent, early_ = float(d.roll.tail(60).mean()), float(d.roll.head(60).mean())
drift = 'grown more bullish' if recent > early_ else 'cooled'
titled(fig, f'Has sentiment toward {TS_TICKER} shifted over time? It has {drift}',
       'Rolling 90-day share of active sentiment days that were positive. '
       'Shading marks stretches where negative days dominated', height=440)
fig.show()
print(f'First 60 readings {early_:.1%} positive → last 60 {recent:.1%}')

First 60 readings 77.3% positive → last 60 59.2%


**Finding.** Sentiment toward a single company is far from stationary, moving through extended
bullish and bearish regimes rather than oscillating around a fixed level. That matters for
interpretation: a positive reading during a broadly bullish stretch carries less information
than the same reading when the mood is otherwise negative.

---

## Q10 · Did chatter move before or after large price moves?

*Dashboard: Key Moments page*

This is the causal-ordering question, and correlation cannot answer it. Taking one abnormal
price move — beyond two standard deviations — and plotting sentiment across the surrounding
window shows directly whether discussion anticipated the move or reported it.

In [16]:
km = key_moves.copy()
for cc in ('move_date','sent_date'):
    km[cc] = pd.to_datetime(km[cc])
km = km[km.ticker.isin(ALL_TICKERS)]
if 'predicted' in km.columns:
    km = km[km.predicted == True]

mv = km.sort_values('zscore', key=abs, ascending=False).iloc[0]
w0, w1 = mv.move_date - pd.Timedelta(days=30), mv.move_date + pd.Timedelta(days=15)
pw = prices[(prices.ticker == mv.ticker) & (prices.Date >= w0) & (prices.Date <= w1)].sort_values('Date')
sw = signals[(signals.ticker == mv.ticker) & (signals.date >= w0) & (signals.date <= w1)].sort_values('date')

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    row_heights=[0.56, 0.44],
                    subplot_titles=('', 'Sentiment across the same window'))
ref = float(pw.Close.iloc[0]); pct = (pw.Close - ref)/ref*100
fig.add_trace(go.Scatter(x=pw.Date, y=pct, mode='lines',
                         line=dict(color=PRICE, width=2.5), name='Price',
                         hovertemplate='%{x|%d %b %Y}<br>%{y:+.1f}%<extra></extra>'),
              row=1, col=1)
fig.add_trace(go.Scatter(x=pw.Date, y=np.where(pct >= 0, pct, 0), fill='tozeroy',
                         mode='none', fillcolor='rgba(0,114,178,0.16)',
                         showlegend=False, hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Scatter(x=pw.Date, y=np.where(pct < 0, pct, 0), fill='tozeroy',
                         mode='none', fillcolor='rgba(213,94,0,0.16)',
                         showlegend=False, hoverinfo='skip'), row=1, col=1)

sv = sw[SIG].fillna(0)
bc = np.where(sv >= SENTIMENT_THRESHOLD, POS,
      np.where(sv <= -SENTIMENT_THRESHOLD, NEG, NEU))
fig.add_trace(go.Bar(x=sw.date, y=sv, marker_color=bc, marker_line_width=0,
                     showlegend=False,
                     hovertemplate='%{x|%d %b %Y}<br>%{y:.3f}<extra></extra>'), row=2, col=1)

for rw in (1, 2):
    fig.add_vline(x=mv.move_date, line_color=GOLD, line_width=3, row=rw, col=1)
    fig.add_vline(x=mv.sent_date, line_color=HIGHLIGHT, line_width=2, line_dash='dash',
                  row=rw, col=1)
for colour, width, dash, name in [
        (GOLD, 4, None, 'Day of the price move'),
        (HIGHLIGHT, 3, 'dash', f'Sentiment fired {int(mv.days_before)}d earlier'),
        (POS, 10, None, 'Positive sentiment'), (NEG, 10, None, 'Negative sentiment')]:
    fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
                             line=dict(color=colour, width=width, dash=dash), name=name),
                  row=1, col=1)

fig.update_yaxes(title='% change from window start', row=1, col=1)
fig.update_yaxes(title='Sentiment score', row=2, col=1)
fig.update_xaxes(tickformat='%d %b', row=1, col=1)
fig.update_xaxes(tickformat='%d %b', row=2, col=1)
fig.update_layout(bargap=0.15, margin=dict(b=130),
                  legend=dict(orientation='h', yanchor='top', y=-0.14, x=0,
                              itemwidth=60, itemsizing='constant'))
titled(fig,
       f'Did chatter move first? Sentiment fired {int(mv.days_before)}d before '
       f'{mv.ticker} moved {mv.return_pct:+.1f}%',
       f'{mv.move_date:%d %B %Y} · signal strength {mv.sentiment:+.3f} · '
       f'z-score {mv.zscore:+.1f} · sources: {mv.get("sources_active","unknown")}',
       height=760)
fig.show()

if 'predicted' in key_moves.columns:
    share = key_moves.predicted.mean()
    print(f'Across all recorded moves, sentiment preceded the move correctly '
          f'{share:.1%} of the time.')

Across all recorded moves, sentiment preceded the move correctly 25.4% of the time.


**Finding.** In this event the sentiment signal precedes the price move rather than following
it, which is the ordering required for the signal to be usable. One event is an illustration
rather than proof — the aggregate figure printed above is the stronger evidence, and the
dashboard allows any recorded move to be inspected the same way.

---

## Q11 · Which source supplies the coverage for each company?

*Dashboard: Sources page, first figure*

The six feeds differ enormously in volume, and a company covered mainly by wire services is
a different proposition from one driven by retail forums. This shows the composition per
ticker.

In [17]:
top25 = (source_attr.groupby('ticker')['item_count'].sum()
         .sort_values(ascending=False).head(25).index)
piv = source_attr[source_attr.ticker.isin(top25)].pivot_table(
    index='ticker', columns='source', values='item_count', fill_value=0)
share = piv.div(piv.sum(axis=1), axis=0) * 100
share = share.loc[piv.sum(axis=1).sort_values(ascending=False).index]

fig = go.Figure()
for col in share.columns:
    fig.add_trace(go.Bar(x=share.index, y=share[col], name=col,
                         marker_color=SOURCE_COLORS.get(col, CONTEXT), marker_line_width=0,
                         hovertemplate=f'<b>{col}</b><br>%{{x}}'
                                       f'<br>%{{y:.1f}}% of coverage<extra></extra>'))
fig.update_layout(barmode='stack', bargap=0.22,
                  legend=dict(orientation='h', y=-0.30, x=0))
fig.update_yaxes(title="Share of that ticker's coverage (%)", ticksuffix='%')
fig.update_xaxes(title='', tickangle=-45)
dominant = share.mean().idxmax()
titled(fig, f'Which source supplies the most coverage? {dominant}',
       f'Coverage composition for the {len(share)} most-discussed tickers', height=580)
fig.show()
print(share.mean().sort_values(ascending=False).round(1).to_string())

source
gdelt                      53.3
hn                         25.1
reddit_technology           8.2
reddit_investing            4.3
reddit_wallstreetbets       4.1
reddit_stocks               3.4
edgar_8k                    0.7
reddit_SecurityAnalysis     0.5
stocktwits                  0.4


**Finding.** Composition is far from uniform. One feed dominates the aggregate while the
others contribute unevenly across companies, meaning a sentiment score for one ticker is not
built from the same mixture of voices as another — a caveat that applies to every
cross-company comparison in this notebook.

---

## Q12 · Which sources word things most strongly?

*Dashboard: Sources page, second figure*

Average absolute sentiment measures how forcefully a source phrases things. It is explicitly
*not* a measure of accuracy, and separating the two is the point of the question.

In [18]:
quality = (source_attr.groupby('source')
           .agg(avg_abs=('avg_sentiment', lambda x: float(x.abs().mean())),
                avg_signed=('avg_sentiment','mean'), items=('item_count','sum'))
           .reset_index().sort_values('avg_abs'))

fig = go.Figure()
for lean, colour, label in [(True, POS, 'Leans net positive'),
                            (False, NEG, 'Leans net negative')]:
    d = quality[(quality.avg_signed >= 0) == lean]
    if d.empty:
        fig.add_trace(go.Bar(y=[None], x=[None], orientation='h',
                             name=f'{label} (none)', marker_color=colour,
                             marker_line_width=0, hoverinfo='skip'))
        continue
    fig.add_trace(go.Bar(y=d.source, x=d.avg_abs, orientation='h', name=label,
                         marker_color=colour, marker_line_width=0,
                         text=[f'{v:.3f}' for v in d.avg_abs], textposition='outside',
                         textfont=dict(size=12, color=MUTED),
                         customdata=np.stack([d['items'], d.avg_signed], axis=-1),
                         hovertemplate='<b>%{y}</b><br>strength %{x:.3f}'
                                       '<br>net lean %{customdata[1]:+.3f}'
                                       '<br>%{customdata[0]:,} items<extra></extra>'))
fig.update_xaxes(title='Average absolute sentiment score')
fig.update_yaxes(title='')
fig.update_layout(legend=dict(orientation='h', yanchor='top', y=-0.22, x=0,
                              itemwidth=40, itemsizing='constant'),
                  margin=dict(b=110))
strongest = quality.iloc[-1]
titled(fig, f"Which source words things most strongly? {strongest['source']}",
       'Bar length is how forcefully a source phrases things, not how often it is right',
       height=500)
fig.show()
print(quality.round(4).to_string(index=False))

                 source  avg_abs  avg_signed  items
               edgar_8k   0.0000      0.0000   3730
                  gdelt   0.0000      0.0000 241536
                     hn   0.1014      0.0188 131382
          reddit_stocks   0.1240      0.0713  26768
  reddit_wallstreetbets   0.1242      0.0689  32316
reddit_SecurityAnalysis   0.1558      0.0976   3747
       reddit_investing   0.1649      0.1111  23421
      reddit_technology   0.1782      0.0996  43164
             stocktwits   0.5144      0.5144   2312


**Finding.** The most emphatic source is not the largest, and emphasis tells us nothing about
correctness — a feed can be consistently forceful and consistently wrong. This is why source
weights in the pipeline are calibrated from measured hit rates rather than from either volume
or expressiveness.

---

## Q13 · How does sentiment track price for a single company?

*Dashboard: Sentiment vs Price page*

The most granular view: one company, sentiment and price side by side over the full period.
Useful for seeing where the two visibly move together and where they decouple entirely.

In [19]:
OVERVIEW_TICKER = 'NVDA'

sig_a = signals[(signals.ticker == OVERVIEW_TICKER) & signals[SIG].notna()].sort_values('date')
p_a   = prices[prices.ticker == OVERVIEW_TICKER].sort_values('Date')

fig = make_subplots(specs=[[{'secondary_y': True}]])
for mask, colour, label in [
        (sig_a[SIG] >= SENTIMENT_THRESHOLD,  POS, 'Positive sentiment'),
        (sig_a[SIG] <= -SENTIMENT_THRESHOLD, NEG, 'Negative sentiment'),
        ((sig_a[SIG] > -SENTIMENT_THRESHOLD) & (sig_a[SIG] < SENTIMENT_THRESHOLD),
         NEU, 'Neutral')]:
    if not mask.any():
        continue
    fig.add_trace(go.Bar(x=sig_a.date[mask], y=sig_a[SIG][mask], marker_color=colour,
                         marker_line_width=0, name=label, opacity=0.7,
                         hovertemplate='%{x|%d %b %Y}<br>%{y:.3f}<extra></extra>'),
                  secondary_y=False)
fig.add_trace(go.Scatter(x=sig_a.date, y=sig_a[SIG].rolling(30, min_periods=1).mean(),
                         mode='lines', line=dict(color=INK, width=2.5),
                         name='30-day average'), secondary_y=False)
fig.add_trace(go.Scatter(x=p_a.Date, y=p_a.pct_7d, mode='lines',
                         line=dict(color=PRICE, width=1.5),
                         name='7-day price change'), secondary_y=True)
fig.update_yaxes(title_text='Sentiment score', secondary_y=False)
fig.update_yaxes(title_text='7-day price change (%)', secondary_y=True, showgrid=False)
fig.update_layout(barmode='relative', bargap=0.1, hovermode='x unified',
                  legend=dict(orientation='h', y=1.03, x=0, yanchor='bottom'))
avg = float(sig_a[SIG].mean())
mood = ('leaned positive' if avg > 0.02 else
        'leaned negative' if avg < -0.02 else 'stayed near neutral')
titled(fig, f'How did sentiment toward {OVERVIEW_TICKER} behave? It {mood}',
       f'{len(sig_a):,} days carried a reading · average score {avg:+.3f}', height=480)
fig.show()

**Finding.** At daily resolution the two series are visibly noisy against each other, which is
the honest picture — the relationship established statistically earlier in this notebook is
real but weak, and would not be obvious from eyeballing a single company. The rolling average
makes the underlying drift legible where the daily bars do not.

---

# Conclusions

**Online chatter carries a real but modest edge.** Positive sentiment calls direction
correctly more often than chance across tens of thousands of signals. The margin is small, as
it must be — a large and obvious edge would already have been competed away.

**The edge sits on the positive side.** Negative sentiment did not clear chance in this
dataset. Rather than read that as a contrarian trading rule, which would contradict
established research on negative news, it more likely reflects the limits of the measurement:
a smaller sample, a sentiment model untuned for financial language, and headlines scored
without their articles.

**It is conditional, not universal.** Predictive strength varies by company, and it has
decayed since 2021. Treating sentiment as uniformly useful discards most of what makes it
usable.

**Volume and reliability are different properties.** The most emphatic and the highest-volume
sources are not the most accurate, which is why weights are calibrated from hit rates.

**It survives execution, but not comfortably.** Both signal-driven strategies beat the passive
benchmarks after costs while trailing the index for long stretches, and the return comes from
a handful of large winners rather than a high win rate.

### Limitations

Sentiment is scored from headlines and post titles rather than full article text. VADER is a
general-purpose model, not tuned for financial language where "beat expectations" and "missed
estimates" carry meanings it does not know. Backtests exclude slippage and assume fills at the
close. Coverage is heavily skewed toward a handful of large companies. Matching company names
in free text produces false positives that survive filtering. Several results rest on modest
samples once the data is split, and the confidence intervals are part of the finding rather
than decoration. This is one historical period in one market, and the post-2021 decay is
itself evidence that these relationships do not hold still.

### Future work

**Replacing VADER with FinBERT is the highest-value change.** FinBERT is fine-tuned on
analyst reports and financial news and reads context rather than matching a lexicon, which
would settle whether the negative-signal result is a real effect or a measurement artifact.
Scoring full articles and using FinBERT's confidence scores to drop uncertain readings would
both help. The cost is compute: VADER runs locally in minutes, FinBERT needs GPU inference
across eleven years of six sources.

Also worth pursuing: intraday resolution, explicit market-regime conditioning given the
post-2021 decay, and testing markets outside the US.

### Interactive dashboard

Every figure above appears in the [Streamlit dashboard](https://share.streamlit.io), deployed
from `stock_tracking/app.py`, with sector and date filters and a what-if calculator for
arbitrary investment amounts.